<a href="https://colab.research.google.com/github/ekaratnida/Data-Engineer/blob/main/de/Lab1/DataFormat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AVRO

In [ ]:
!pip install fastavro

In [3]:
#pip install fastavro
from fastavro import writer, reader, parse_schema

schema = {
    'doc': 'A weather reading.',
    'name': 'Weather',
    'namespace': 'test',
    'type': 'record',
    'fields': [
        {'name': 'station', 'type': 'string'},
        {'name': 'time', 'type': 'long'},
        {'name': 'temp', 'type': 'int'},
    ],
}

parsed_schema = parse_schema(schema)

# 'records' can be an iterable (including generator)
records = [
    {u'station': u'011990-99999', u'temp': 0, u'time': 1433269388},
    {u'station': u'011990-99999', u'temp': 22, u'time': 1433270389},
    {u'station': u'011990-99999', u'temp': -11, u'time': 1433273379},
    {u'station': u'012650-99999', u'temp': 111, u'time': 1433275478},
    {u'station': u'012650-11', u'temp': 211, u'time': 1433275479, u'officer': u'Peter'},
]

# Writing
with open('weather.avro', 'wb') as out:
  writer(out, parsed_schema, records)


In [ ]:
with open('weather.avro', 'rb') as fo:
  for record in reader(fo):
    print(record)

#Parquet

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/ekaratnida/Data-Engineer/refs/heads/main/de/Lab1/parquet/bank.csv")
df.head()

In [7]:
df.to_parquet("bank.parquet")

In [ ]:
df = pd.read_parquet("bank.parquet")
df.head()

#JSON

In [46]:
from json import loads, dumps
df = pd.read_json("https://raw.githubusercontent.com/ekaratnida/Data-Engineer/refs/heads/main/de/Lab1/json/sample.json")
result = df.to_json()
parsed = loads(result)
print(parsed)
print(parsed['team'])
print(parsed['team']['0'])
#d = dumps(parsed, indent=4)

{'team': {'0': {'name': 'Peter', 'age': 20}, '1': {'name': 'John', 'age': 30}}}
{'0': {'name': 'Peter', 'age': 20}, '1': {'name': 'John', 'age': 30}}
{'name': 'Peter', 'age': 20}


#Basic ETL

In [47]:
import pandas as pd
import numpy as np

def extract_data():

    raw_data = {
        'Transaction_ID': [101, 102, 103, 104],
        'User_ID': [5001, 5002, 5001, 5003],
        'Product': [' laptop ', 'Smartphone', ' LAPTOP ', 'Headphones'],
        'Price': [1200.00, 800.00, 1200.00, np.nan],
        'Quantity': [1, 2, 1, 1],
        'Purchase_Date': ['2026/05/20', '20-05-2026', '2026-05-21', '2026-05-22']
    }
    df = pd.DataFrame(raw_data)
    print("--- Extracted Raw Data ---")
    print(df, "\n")
    return df

def transform_data(df):

    print("--- Starting Transformation ---")

    df = df.dropna(subset=['Price'])

    df['Product'] = df['Product'].str.strip().str.upper()

    df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'], errors='coerce').dt.strftime('%Y-%m-%d')

    df['Total_Spent'] = df['Price'] * df['Quantity']

    df = df.drop_duplicates()

    print(df, "\n")

    return df

def load_data(df, target_destination):

    df.to_csv(target_destination, index=False)

    print(f"--- Load Complete! Data successfully saved to: {target_destination} ---")

if __name__ == "__main__":

    # 1. Extract
    raw_df = extract_data()

    # 2. Transform
    transformed_df = transform_data(raw_df)

    # 3. Load
    load_data(transformed_df, 'warehouse_sales_fact.csv')

--- Extracted Raw Data ---
   Transaction_ID  User_ID     Product   Price  Quantity Purchase_Date
0             101     5001     laptop   1200.0         1    2026/05/20
1             102     5002  Smartphone   800.0         2    20-05-2026
2             103     5001     LAPTOP   1200.0         1    2026-05-21
3             104     5003  Headphones     NaN         1    2026-05-22 

--- Starting Transformation ---
   Transaction_ID  User_ID     Product   Price  Quantity Purchase_Date  \
0             101     5001      LAPTOP  1200.0         1    2026-05-20   
1             102     5002  SMARTPHONE   800.0         2           NaN   
2             103     5001      LAPTOP  1200.0         1           NaN   

   Total_Spent  
0       1200.0  
1       1600.0  
2       1200.0   

--- Load Complete! Data successfully saved to: warehouse_sales_fact.csv ---


/tmp/ipykernel_661/2456041322.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Product'] = df['Product'].str.strip().str.upper()
/tmp/ipykernel_661/2456041322.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'], errors='coerce').dt.strftime('%Y-%m-%d')
/tmp/ipykernel_661/2456041322.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the